[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_13_Multimodal_AI_Vision_Text.ipynb)

# 🖼️ Lesson 13 — Multimodal AI: Vision + Text

**Course:** Becoming a Practical AI Engineer  
**Lesson:** 13 of 15  
**Phase:** 2 — Production Skills

---

## 🎯 What You'll Learn Today

Until now, every agent you've built has been **text-in, text-out**. But the real world is full of images — screenshots, diagrams, invoices, charts, scanned documents, and photos. Today you unlock the ability to **see**.

By the end of this lesson you will:
- Understand what "multimodal" means and why it matters for agents
- Send images to Claude and extract structured information
- Parse visual documents (receipts, charts, tables) into usable data
- Build a **Vision + RAG pipeline** — index images by their text descriptions and retrieve them semantically

---

## 🧠 Concept First: What Is Multimodal AI?

A **unimodal** model handles one type of input — text. A **multimodal** model handles multiple types — text, images, audio, video.

Claude (claude-sonnet-4-6, claude-opus-4-6) is multimodal for **vision**: it can read images alongside text in the same message.

### How it works technically

When you call the Anthropic API with an image, the `content` field of your message changes from a single string to a **list of content blocks**:

```
Text-only message:
  content = "What is the capital of France?"

Multimodal message:
  content = [
    { type: "image", source: { type: "base64", data: "...", media_type: "image/jpeg" } },
    { type: "text", text: "What does this image show?" }
  ]
```

Claude processes both streams together — it "sees" the image and reasons about it in the same pass as the text.

### Two ways to send an image

| Method | When to use | Notes |
|--------|------------|-------|
| **Base64** | Local files, generated images, private data | Encode the raw bytes as a base64 string — bigger payload |
| **URL** | Public images already on the web | Just pass the URL — Claude fetches it internally |

### What Claude can see vs. what it cannot

✅ Claude CAN: Read text in images (OCR-like), describe scenes, identify objects, analyze charts and graphs, compare multiple images, parse tables, read code screenshots

❌ Claude CANNOT: Identify specific real people by face, read very low-resolution text, detect content below ~20px

---

## ⚙️ Setup — Install & Configure

**One-time Colab secret setup:**  
1. Click the 🔑 **Secrets** icon in the left sidebar  
2. Add a secret named `ANTHROPIC_API_KEY` with your key  
3. Toggle "Notebook access" to ON  

Then run the cell below.

In [ ]:
# Install dependencies
!pip install anthropic requests Pillow chromadb -q

import anthropic
import base64
import requests
from io import BytesIO
from PIL import Image
import json

# Load API key from Colab Secrets
from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY')

client = anthropic.Anthropic(api_key=api_key)
print("✅ Client ready!")

---

## 📦 Helper: Load an Image as Base64

This utility converts any image URL or local file path into the base64-encoded format Claude expects.

In [ ]:
def image_url_to_base64(url: str) -> tuple[str, str]:
    """
    Fetches an image from a URL and returns (base64_data, media_type).
    """
    response = requests.get(url)
    response.raise_for_status()
    
    # Detect media type from Content-Type header
    content_type = response.headers.get('Content-Type', 'image/jpeg')
    media_type = content_type.split(';')[0].strip()  # e.g. "image/jpeg"
    
    # Encode bytes to base64 string
    b64_data = base64.standard_b64encode(response.content).decode('utf-8')
    
    return b64_data, media_type


def ask_about_image_url(image_url: str, question: str, model: str = "claude-sonnet-4-6") -> str:
    """
    Send a public image URL + question to Claude. Simplest possible multimodal call.
    """
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "url",      # <-- simplest method: just give the URL
                            "url": image_url,
                        },
                    },
                    {
                        "type": "text",
                        "text": question
                    }
                ],
            }
        ],
    )
    return response.content[0].text


def ask_about_image_base64(image_url: str, question: str, model: str = "claude-sonnet-4-6") -> str:
    """
    Fetch an image, encode it as base64, then send to Claude.
    Use this for private/local images or when URL access isn't available.
    """
    b64_data, media_type = image_url_to_base64(image_url)
    
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",          # <-- encode the raw bytes
                            "media_type": media_type,  # e.g. "image/jpeg"
                            "data": b64_data,          # base64 string
                        },
                    },
                    {
                        "type": "text",
                        "text": question
                    }
                ],
            }
        ],
    )
    return response.content[0].text

print("✅ Helpers loaded!")

---

## 🔍 Section 1: Basic Image Understanding

Let's start simple — describe a real image.

In [ ]:
# A publicly available chart image (NASA's global temperature anomaly graph)
CHART_URL = "https://climate.nasa.gov/system/charts/15_a_global_temperature_chart.png"

# 💡 EXPERIMENT: Try any public image URL here!
# e.g., a screenshot, a product photo, a diagram from a blog post

description = ask_about_image_url(
    image_url=CHART_URL,
    question="Describe what this image shows. What kind of chart is it, and what trend does it display?"
)

print("Claude's description:")
print("-" * 50)
print(description)

### 🧠 What just happened?

Claude received the image bytes + your text question in a single API call. It processed both modalities simultaneously — not sequentially. There's no separate "image description" step; the model reasons about image and text together.

This is fundamentally different from older pipelines where you'd run an image captioning model first, then feed the caption to a language model.

---

## 📄 Section 2: Visual Document Parsing

**Visual document parsing** means extracting structured data from images of documents — receipts, invoices, business cards, screenshots of tables, etc.

This is where multimodal AI earns its keep in production. Instead of a fragile OCR pipeline followed by regex parsing, you can just ask Claude to extract the data directly into JSON.

### The pattern:
1. Give Claude the image
2. Ask for JSON output with a specific schema
3. Parse the JSON in your code

Let's try with a sample receipt image.

In [ ]:
import json

def extract_structured_data_from_image(image_url: str, schema_description: str) -> dict:
    """
    Extract structured JSON data from an image using Claude.
    
    This is the core pattern for visual document parsing:
    - Give Claude the image
    - Describe the JSON schema you want
    - Parse the response
    """
    prompt = f"""Analyze this image and extract the data as JSON.

Schema description: {schema_description}

IMPORTANT: Return ONLY valid JSON, no explanation text. If a field is not visible or not applicable, use null."""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "url",
                            "url": image_url,
                        },
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ],
            }
        ],
    )
    
    raw_text = response.content[0].text.strip()
    
    # Strip markdown code fences if present
    if raw_text.startswith("```"):
        raw_text = raw_text.split("```")[1]
        if raw_text.startswith("json"):
            raw_text = raw_text[4:]
    
    return json.loads(raw_text)


# Sample receipt image (a publicly available demo receipt)
RECEIPT_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5c/Store_receipt_2009.jpg/220px-Store_receipt_2009.jpg"

schema = """
{
  "store_name": "string",
  "date": "string (YYYY-MM-DD format if possible)",
  "items": [
    {"name": "string", "quantity": number or null, "price": number}
  ],
  "subtotal": number or null,
  "tax": number or null,
  "total": number or null,
  "payment_method": "string or null"
}
"""

print("📄 Parsing receipt image...")
try:
    extracted = extract_structured_data_from_image(RECEIPT_URL, schema)
    print("✅ Extracted data:")
    print(json.dumps(extracted, indent=2))
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print("Raw response may need adjustment — try adding a retry with stricter prompt")

### 💡 Why This Matters for Agents

Imagine an **expense agent** that:
1. Receives receipt photos from employees via Slack
2. Calls `extract_structured_data_from_image` on each
3. Validates the data with Pydantic (Lesson 10)
4. Writes to a spreadsheet or accounting system

What used to require a dedicated OCR service + custom parsing rules becomes a single Claude API call. This is the compounding leverage of combining the skills you've built.

---

## 📊 Section 3: Chart & Diagram Analysis

Charts and diagrams are especially hard for traditional tools — the meaning is in the visual layout, not just the text. Claude can reason about what charts are **saying**, not just what text appears in them.

In [ ]:
def analyze_chart(image_url: str) -> dict:
    """
    Deep analysis of a chart or data visualization.
    Returns structured insights, not just a description.
    """
    prompt = """Analyze this chart or data visualization thoroughly.

Return your analysis as JSON with this structure:
{
  "chart_type": "bar/line/pie/scatter/table/diagram/other",
  "title": "chart title if visible",
  "x_axis": "what the x axis represents",
  "y_axis": "what the y axis represents",
  "key_insight": "the single most important takeaway from this chart",
  "trend": "describe the main trend (increasing, decreasing, cyclical, etc.)",
  "anomalies": ["list any outliers or notable data points"],
  "data_summary": "2-3 sentence plain language summary of what this data shows"
}

Return ONLY valid JSON."""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {"type": "url", "url": image_url},
                    },
                    {"type": "text", "text": prompt}
                ],
            }
        ],
    )
    
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    
    return json.loads(raw)


# A publicly available bar chart example
CHART_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Tonal_ranges_of_voices_in_choral_music.png/320px-Tonal_ranges_of_voices_in_choral_music.png"

# 💡 EXPERIMENT: Paste the URL of any chart from a news article, a Wikipedia page, or your own work

print("📊 Analyzing chart...")
try:
    analysis = analyze_chart(CHART_URL)
    print("✅ Chart analysis:")
    print(json.dumps(analysis, indent=2))
    
    print("\n🔑 Key Insight:")
    print(analysis.get('key_insight', 'N/A'))
except Exception as e:
    print(f"Error: {e}")

---

## 🔀 Section 4: Multi-Image Comparison

Claude can handle **multiple images in a single message**. This is powerful for:
- Before/after comparison
- Product variant analysis  
- Error diagnosis (screenshot + reference image)

The API design is simple: just add more image content blocks to the list.

In [ ]:
def compare_images(image_url_1: str, image_url_2: str, comparison_question: str) -> str:
    """
    Send two images to Claude for comparison in a single API call.
    
    Key insight: the content list can have as many image blocks as needed.
    Claude sees all of them at once.
    """
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Here are two images (Image 1 and Image 2):"},
                    {
                        "type": "image",
                        "source": {"type": "url", "url": image_url_1},
                    },
                    {"type": "text", "text": "Image 2:"},
                    {
                        "type": "image",
                        "source": {"type": "url", "url": image_url_2},
                    },
                    {
                        "type": "text",
                        "text": comparison_question
                    }
                ],
            }
        ],
    )
    return response.content[0].text


# Two related Wikipedia images to compare
IMAGE_1 = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Bikesgray.jpg/320px-Bikesgray.jpg"
IMAGE_2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/44/Bike_in_studio_2.jpg/320px-Bike_in_studio_2.jpg"

# 💡 EXPERIMENT: Use before/after UI screenshots, two versions of a chart, or two product photos

result = compare_images(
    IMAGE_1, IMAGE_2,
    "Compare these two images. What are the key differences and similarities?"
)

print("🔀 Comparison result:")
print(result)

---

## 🏗️ Section 5: Vision + RAG — The Capstone Pattern

This is the most architecturally interesting part of today's lesson.

### The Problem

You have a library of 1,000 images (product manuals, architecture diagrams, charts). A user asks: *"Show me the diagram that explains the authentication flow."*

You can't embed images directly into a vector store — ChromaDB stores text embeddings. So how do you make images semantically searchable?

### The Solution: Vision → Description → Embed

```
INDEXING PIPELINE:
  Image → Claude describes it → Text description → Embed → Store in ChromaDB
  (with original image URL stored as metadata)

RETRIEVAL PIPELINE:
  User query → Embed query → ChromaDB similarity search → Top-K descriptions
  → Fetch original images via stored URLs → Claude answers using the images
```

This is **Vision + RAG**: the retrieval layer is text-based, but the reasoning layer uses the actual images.

### Why not just search descriptions forever?

Because descriptions lose detail. When you retrieve the image and feed it to Claude at query time, Claude can answer questions about fine-grained details that the description might have missed.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# ──────────────────────────────────────────
# STEP 1: Define our image library
# In production, these would be your actual documents/diagrams
# For this demo we use publicly available Wikipedia images
# ──────────────────────────────────────────

IMAGE_LIBRARY = [
    {
        "id": "img_001",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/Sql_logo.png/320px-Sql_logo.png",
        "label": "SQL Database Logo"
    },
    {
        "id": "img_002",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg",
        "label": "Ant close-up photo"
    },
    {
        "id": "img_003",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png",
        "label": "Color gradient transparency demo"
    },
    {
        "id": "img_004",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Tonal_ranges_of_voices_in_choral_music.png/320px-Tonal_ranges_of_voices_in_choral_music.png",
        "label": "Music voice tonal range chart"
    },
    {
        "id": "img_005",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Bikesgray.jpg/320px-Bikesgray.jpg",
        "label": "Bicycles parked"
    },
]

print(f"📚 Image library defined: {len(IMAGE_LIBRARY)} images")
for img in IMAGE_LIBRARY:
    print(f"  [{img['id']}] {img['label']}")

In [ ]:
def generate_image_description(image_url: str) -> str:
    """
    Generate a rich text description of an image for indexing.
    This is the bridge between the visual world and the text embedding world.
    """
    prompt = """Describe this image in detail for a search index. Include:
- What the image shows (objects, people, text, charts, diagrams)
- The type of image (photo, diagram, logo, chart, screenshot, etc.)
- Any text visible in the image
- The subject matter / domain (technology, nature, science, business, etc.)
- Colors and visual style if relevant

Write 3-5 sentences. Be specific and include terms someone might search for."""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",  # Use Haiku for indexing — fast + cheap
        max_tokens=256,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {"type": "url", "url": image_url},
                    },
                    {"type": "text", "text": prompt}
                ],
            }
        ],
    )
    return response.content[0].text


# ──────────────────────────────────────────
# STEP 2: Index — generate descriptions + embed
# ──────────────────────────────────────────

print("🔄 Generating descriptions and indexing images...\n")

# Set up ChromaDB with default embedding function
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="image_library",
    embedding_function=embedding_functions.DefaultEmbeddingFunction()  # Uses local model
)

descriptions = []
for img in IMAGE_LIBRARY:
    print(f"  Processing {img['id']}: {img['label']}...")
    desc = generate_image_description(img['url'])
    descriptions.append(desc)
    print(f"  → {desc[:100]}...\n")

# Add all to ChromaDB
# Documents = text descriptions (what gets embedded)
# Metadata = original image URL + label (what gets retrieved)
collection.add(
    documents=descriptions,
    metadatas=[{"url": img['url'], "label": img['label']} for img in IMAGE_LIBRARY],
    ids=[img['id'] for img in IMAGE_LIBRARY]
)

print(f"\n✅ Indexed {len(IMAGE_LIBRARY)} images into ChromaDB!")

In [ ]:
def vision_rag_query(user_question: str, n_results: int = 2) -> str:
    """
    Full Vision + RAG pipeline:
    1. Embed the user's query
    2. Find the most similar image descriptions in ChromaDB
    3. Fetch the actual images
    4. Ask Claude to answer the question using the retrieved images
    """
    
    # ── Step 1: Semantic search over image descriptions ──
    results = collection.query(
        query_texts=[user_question],
        n_results=n_results
    )
    
    retrieved = []
    for i in range(len(results['ids'][0])):
        retrieved.append({
            "id": results['ids'][0][i],
            "url": results['metadatas'][0][i]['url'],
            "label": results['metadatas'][0][i]['label'],
            "description": results['documents'][0][i],
            "distance": results['distances'][0][i]
        })
    
    print(f"🔍 Retrieved {len(retrieved)} images for query: '{user_question}'")
    for r in retrieved:
        print(f"   [{r['id']}] {r['label']} (distance: {r['distance']:.3f})")
    print()
    
    # ── Step 2: Build a multimodal message with the retrieved images ──
    content_blocks = []
    content_blocks.append({
        "type": "text",
        "text": f"I'm going to show you {len(retrieved)} retrieved images. Please answer this question using them: {user_question}"
    })
    
    for i, r in enumerate(retrieved, 1):
        content_blocks.append({
            "type": "text",
            "text": f"Image {i} (label: {r['label']}):"
        })
        content_blocks.append({
            "type": "image",
            "source": {"type": "url", "url": r['url']}
        })
    
    # ── Step 3: Claude reasons over actual images ──
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": content_blocks
            }
        ]
    )
    
    return response.content[0].text


# ──────────────────────────────────────────
# Test the Vision + RAG pipeline
# ──────────────────────────────────────────

queries = [
    "Show me something related to databases or data storage",
    "I'm looking for an image of an insect or bug",
    "Find me a chart or visualization about music"
]

# 💡 EXPERIMENT: Change these queries to see how retrieval + reasoning changes

for query in queries:
    print("=" * 60)
    answer = vision_rag_query(query, n_results=1)
    print(f"💬 Answer: {answer[:300]}...\n")

---

## 🏭 Section 6: Production Considerations

Now that you've built the core patterns, let's think about what changes when you go to production.

In [ ]:
import time

# ──────────────────────────────────────────
# 1. Image Size Optimization
# Claude has image size limits. Large images = high latency + cost.
# Best practice: resize images before sending.
# ──────────────────────────────────────────

def resize_image_for_claude(image_url: str, max_size: int = 1568) -> tuple[str, str]:
    """
    Download an image, resize it to fit within Claude's optimal dimensions,
    and return as base64.
    
    Claude's limits:
    - Max image size: 5MB
    - Optimal: ≤1568px on the longest side (claude.ai recommendation)
    - Minimum meaningful: ~50px
    """
    response = requests.get(image_url)
    img = Image.open(BytesIO(response.content))
    
    original_size = img.size
    
    # Resize if needed (preserving aspect ratio)
    if max(img.size) > max_size:
        img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    
    print(f"📐 Resized: {original_size} → {img.size}")
    
    # Convert to JPEG bytes
    buffer = BytesIO()
    img.convert('RGB').save(buffer, format='JPEG', quality=85)
    b64 = base64.standard_b64encode(buffer.getvalue()).decode('utf-8')
    
    return b64, "image/jpeg"


# ──────────────────────────────────────────
# 2. Cost Estimation
# Image tokens are calculated differently from text tokens
# Claude charges for images based on their pixel dimensions
# ──────────────────────────────────────────

def estimate_image_tokens(width: int, height: int) -> int:
    """
    Estimate token cost for an image.
    Claude's formula: tiles × 1750 + 85 base tokens
    Each tile is 512×512 pixels.
    """
    tile_size = 512
    tiles_x = (width + tile_size - 1) // tile_size
    tiles_y = (height + tile_size - 1) // tile_size
    total_tiles = tiles_x * tiles_y
    tokens = total_tiles * 1750 + 85
    return tokens


# Show the cost difference between sizes
print("💰 Image token cost by size:")
sizes = [(100, 100), (512, 512), (1024, 768), (1568, 1568), (3000, 2000)]
for w, h in sizes:
    tokens = estimate_image_tokens(w, h)
    cost = tokens * 0.000003  # Approximate cost per token for claude-sonnet-4-6
    print(f"  {w:5d}×{h:5d}px → {tokens:,} tokens ≈ ${cost:.4f} per image")

print("\n💡 Key takeaway: Resize images to ≤1568px to stay in the efficient pricing tier")


# ──────────────────────────────────────────
# 3. Model Selection Strategy for Vision
# ──────────────────────────────────────────

print("""
🤖 Model selection for vision tasks:

  claude-haiku-4-5     → Batch processing, indexing pipelines, simple descriptions
                          Fast, cheap. Good for: "describe this image for search indexing"

  claude-sonnet-4-6    → Production workhorse. Complex reasoning about images.
                          Good for: document parsing, chart analysis, multi-image comparison

  claude-opus-4-6      → Hardest visual reasoning tasks
                          Good for: ambiguous diagrams, multi-step visual problem-solving

Strategy: use Haiku for indexing (run once, cheap), Sonnet for user-facing queries.
""")

---

## 🎯 Mini-Project: Visual Invoice Processor Agent

Bring it all together: build an agent that processes invoice images and extracts structured payment data.

**Stretch goal (try after completing the lesson):** Extend this to handle multiple invoice images and aggregate totals.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List
from datetime import date

# ── Define the data model (from Lesson 10!) ──
class LineItem(BaseModel):
    description: str
    quantity: Optional[float] = None
    unit_price: Optional[float] = None
    total: float

class Invoice(BaseModel):
    vendor_name: str
    invoice_number: Optional[str] = None
    invoice_date: Optional[str] = None
    due_date: Optional[str] = None
    line_items: List[LineItem] = Field(default_factory=list)
    subtotal: Optional[float] = None
    tax_amount: Optional[float] = None
    total_amount: float
    currency: str = "USD"
    payment_terms: Optional[str] = None
    notes: Optional[str] = None


def process_invoice_image(image_url: str) -> Invoice:
    """
    Agent that extracts structured invoice data from an image.
    Combines: Vision API + Pydantic validation + error handling
    """
    
    prompt = f"""You are an invoice processing agent. Extract all financial data from this invoice image.

Return a JSON object matching this exact schema:
{Invoice.model_json_schema()}

Rules:
- Extract ALL line items visible
- Use null for fields not visible in the image
- total_amount is REQUIRED — estimate if not clearly shown
- Dates should be in YYYY-MM-DD format where possible
- Amounts should be numbers (no currency symbols)

Return ONLY valid JSON, no explanation."""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {"type": "url", "url": image_url}
                    },
                    {"type": "text", "text": prompt}
                ]
            }
        ]
    )
    
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    
    # Pydantic validates AND gives us a typed object
    return Invoice.model_validate_json(raw)


# Test with a sample receipt (closest we have to an invoice in public domain)
TEST_IMAGE = "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5c/Store_receipt_2009.jpg/220px-Store_receipt_2009.jpg"

# 💡 EXPERIMENT: Find an invoice image online (or take a photo of a real receipt)
#   and replace TEST_IMAGE with its URL

print("🧾 Processing invoice image...")
try:
    invoice = process_invoice_image(TEST_IMAGE)
    print("\n✅ Extracted Invoice:")
    print(f"  Vendor: {invoice.vendor_name}")
    print(f"  Invoice #: {invoice.invoice_number}")
    print(f"  Date: {invoice.invoice_date}")
    print(f"  Total: {invoice.currency} {invoice.total_amount}")
    if invoice.line_items:
        print(f"  Line items ({len(invoice.line_items)}):")
        for item in invoice.line_items:
            print(f"    - {item.description}: ${item.total}")
    
    print("\n📋 Full Pydantic model:")
    print(invoice.model_dump_json(indent=2))
    
except Exception as e:
    print(f"Error: {e}")
    print("\nThis can happen with low-resolution or unclear images.")
    print("In production: add retry logic with a more explicit prompt.")

---

## 📚 Lesson Recap

### What You Built Today

| Pattern | What it does | When to use it |
|---------|-------------|----------------|
| **URL image** | Pass a public URL directly | Fastest, simplest |
| **Base64 image** | Encode private/local images | Private data, generated images |
| **Structured extraction** | Get JSON from image content | Receipts, forms, tables |
| **Chart analysis** | Extract insights from visualizations | Reporting agents, data pipelines |
| **Multi-image comparison** | Send multiple images at once | Before/after, variants |
| **Vision + RAG** | Index images via descriptions, retrieve and reason | Large image libraries |
| **Vision + Pydantic** | Validated typed data from images | Production parsing agents |

### Key Mental Models

**1. Images are content blocks, not attachments.**  
Claude's API treats images and text as equal citizens in a list of content blocks. You can interleave them freely.

**2. Description bridges visual → searchable.**  
Vector databases store text embeddings. To make images searchable, generate rich text descriptions at index time. At query time, retrieve the actual images.

**3. Use Haiku for indexing, Sonnet/Opus for reasoning.**  
Image processing costs add up fast. Use the cheapest capable model for batch operations.

**4. Combine with Pydantic for production reliability.**  
Visual extraction + schema validation = reliable agent pipelines for document processing.

---

## 🔜 Next Up: Lesson 14 — MCP (Model Context Protocol)

You've now built agents that can see, remember, reason, and retrieve. The next frontier: **making your agents interoperable with any AI system** via the Model Context Protocol. MCP is the emerging standard for packaging tools so that Claude, GPT, Gemini, and any future model can use them without custom integration. You'll build your first MCP server from scratch.

---

*Lesson 13 of 15 — Learn AI with Claude | 2026-05-12*